# Tiền xử lý dữ liệu dự báo thời tiết

Phần thành viên 2, dành cho bài toán phân loại trạng thái ngày kế tiếp tại Hà Nội, TP.HCM và Nha Trang.
Mở notebook từ thư mục dự án của thành viên 2. Cài requirements.txt và nhận dữ liệu từ thành viên 1 để có data/interim/weather_raw.csv trước.
Không có bước huấn luyện hoặc đánh giá thuật toán phân loại trong notebook này.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import numpy as np
ROOT = Path.cwd().resolve()
if not (ROOT / 'preprocessing.py').exists():
    ROOT = ROOT.parent
if not (ROOT / 'preprocessing.py').exists():
    raise RuntimeError('Hãy mở notebook trong thư mục dự án hoặc notebooks')
sys.path.insert(0, str(ROOT))
from preprocessing import (clean_daily, build_supervised, split_by_time,
                                  make_preprocessor, class_distribution,
                                  FEATURE_COLUMNS, NUMERIC_FEATURES, DEFAULT_CONFIG, run)
config = DEFAULT_CONFIG.copy()
config.update(json.loads((ROOT / 'config/preprocessing.json').read_text(encoding='utf-8')))
raw = pd.read_csv(ROOT / 'data/interim/weather_raw.csv')
print('Kích thước dữ liệu nhận từ thành viên 1:', raw.shape)
print(raw.head(3).to_string(index=False))

## Làm sạch và dựng lịch ngày

Giữ áp suất ngoài tập đặc trưng vì đơn vị chưa rõ. Gió km/h được đổi sang m/s.
Các ngày bị thiếu được đưa vào lịch để shift và rolling không nhảy qua ngày lịch.
Chưa điền thiếu ở bước này; nhãn không được suy đoán bằng trung bình hay trung vị.

In [ ]:
daily, audit, invalid_cells, invalid_dates = clean_daily(raw, config)
print(json.dumps(audit, ensure_ascii=False, indent=2))
print(daily.loc[~daily.source_row_present, ['city', 'date']].to_string(index=False))

## Tạo đặc trưng và nhãn ngày kế tiếp

Quy ước: rain >= 1 mm là Mưa. Nếu rain < 1 mm, cloud >= 50% là Nhiều mây, còn lại là Nắng hoặc ít mây.
Nhãn là quy ước áp dụng trên bản ghi ngày của bộ dữ liệu, không phải xác nhận quan trắc độc lập.
Đầu vào lấy từ ngày t hoặc trước đó; y lấy đúng ngày t+1. Ngày t phải có bản ghi nguồn thật.

In [ ]:
dataset, rejected = build_supervised(daily, config)
print('Số mẫu có nhãn ngày kế tiếp:', len(dataset))
print('Số ngày xuất phát không có nhãn để giám sát:', len(rejected))
print(rejected.to_string(index=False))
print(dataset.loc[(dataset.city == 'Nha Trang') &
                  dataset.origin_date.isin(pd.to_datetime(['2020-02-27', '2020-03-01'])),
                  ['origin_date', 'target_date', 'rain_mm', 'cloud_cover_pct', 'label_vi']].to_string(index=False))

## Chia Train Validation Test

Chia khoảng 60/15/25 theo ngày cần dự báo. Cả ba thành phố của cùng một ngày phải nằm trong cùng tập.
Các đặc trưng lịch sử của Validation/Test được phép dùng các quan trắc đã biết đến ngày xuất phát t.

In [ ]:
dataset, splits, boundaries = split_by_time(dataset, config)
print(boundaries.to_string(index=False))
print(class_distribution(dataset).to_string(index=False))

## Điền thiếu và chuẩn hóa chỉ từ Train

SimpleImputer học trung vị từ Train. StandardScaler học trung bình và độ lệch chuẩn của Train sau điền thiếu.
OneHotEncoder học các thành phố của Train. Validation và Test chỉ dùng transform.
Bộ scaled dùng cho Logistic Regression, KNN và SVM; bộ không scaling dùng cho cây quyết định và Random Forest.

In [ ]:
scaled = make_preprocessor(scale=True)
tree = make_preprocessor(scale=False)
X_train_raw = splits['train'][FEATURE_COLUMNS]
scaled.fit(X_train_raw)
tree.fit(X_train_raw)
for split_name, part in splits.items():
    X_scaled = scaled.transform(part[FEATURE_COLUMNS])
    X_tree = tree.transform(part[FEATURE_COLUMNS])
    assert np.isfinite(X_scaled).all() and np.isfinite(X_tree).all()
    print(split_name, 'X shape:', X_scaled.shape, 'y shape:', part.target.shape)
print('Số đặc trưng số trước mã hóa:', len(NUMERIC_FEATURES))
print('Tên các cột sau mã hóa:', scaled.get_feature_names_out().tolist())

## Xuất bộ bàn giao

Lệnh dưới tái tạo CSV, các tập dữ liệu, hai bộ tiền xử lý và metadata bằng cùng quy trình.
Không đổi cấu hình dựa vào kết quả Test. Khi dùng kiểm định chéo ở phần 3, đặt bộ tiền xử lý trong Pipeline và fit lại trong từng fold theo thời gian.

In [ ]:
summary = run(config=config)
print('Các kiểm tra tính toàn vẹn:', summary['checks'])